### Exhaustive 3: Tools (Function Calling) Deep Dive

This notebook breaks down `3-tools.py`, where the model gets live weather data by asking **your code** to run a function.

**The key point:** the model never runs code; it can only write text. "Calling a tool" means the model writes a small request like "please run `get_weather` with these two numbers", and your Python code does the actual running. That's why one tool use needs **two API calls with your code in between**.

##### The whole flow:
```
1. Call 1: client.chat.completions.create(messages, tools)
   The model reads the question + the tool's name/description/parameters
   and replies with a REQUEST, not an answer:
   name='get_weather', arguments='{"latitude":59.9139,"longitude":10.7522}'
            │
            ▼
2. Your code (no model involved):
   json.loads(arguments) -> call_function -> get_weather(**args)
   -> real HTTP request to api.open-meteo.com -> {"temperature_2m": 14.5, ...}
   Append the model's request and the result to `messages`.
            │
            ▼
3. Call 2: client.chat.completions.parse(messages, tools, response_format)
   The model reads the whole history, including the result,
   and writes the answer -> WeatherResponse(temperature=14.5, response='...')
```

- **Section 1**: What the model reads: the tool `description`, not the docstring.
- **Section 2**: Reading the response of call 1.
- **Section 3**: Call 1 did not fetch the weather, so why the rest?
- **Section 4**: `call_function`, `**args`, and the loop line by line.
- **Section 5**: The four messages the second call receives.
- **Section 6**: Call 2, and where `Field()` descriptions go.
- **Section 7**: Checking the final answer against what the model was given.


In [2]:
import json
import os
from pprint import pprint
import requests
from openai import OpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv

In [4]:
load_dotenv()
client = OpenAI()

In [5]:
def get_weather(latitude, longitude):
    """This is a publically available API that returns the weather for a given location."""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m&timezone=auto&wind_speed_unit=ms"
    )
    data = response.json()
    # send the units and timezone too, so the model doesn't have to guess them
    return {
        "current": data["current"],
        "units": data["current_units"],
        "timezone": data["timezone"],
    }

In [6]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current temperature for provided coordinates in celsius.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {"type": "number"},
                    "longitude": {"type": "number"},
                },
                "required": ["latitude", "longitude"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]


#### 1. What the model reads: the tool `description`, not the docstring

**Key point:** the model reads only the JSON request body that the SDK sends, and the Python function `get_weather` is never part of it. Its docstring never leaves your machine. Everything the model knows about the tool comes from the `tools` list, so the text that plays the docstring's role is `"description"`.

Look at the call in the next cell: `create(model=..., messages=..., tools=...)`. `get_weather` isn't passed anywhere. The only link between the model and your function is the **name string** `"get_weather"`, which you wrote in both places.

##### The exact request body of call 1
This is what the SDK sends for the first call. I captured it with a fake HTTP transport, so nothing went to OpenAI. The docstring text ("This is a publically available API...") appears nowhere in it.

<div style="font-size: 0.85em">

```json
{
  "messages": [
    {"role": "system", "content": "You are a helpful weather assistant."},
    {"role": "user", "content": "What's the weather like in Oslo today?"}
  ],
  "model": "gpt-5-nano",
  "tools": [{
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get current temperature for provided coordinates in celsius.",
      "parameters": {
        "type": "object",
        "properties": {"latitude": {"type": "number"}, "longitude": {"type": "number"}},
        "required": ["latitude", "longitude"],
        "additionalProperties": false
      },
      "strict": true
    }
  }]
}
```

</div>

##### What the model uses each part of the tool for
- **`name`**: the label the model writes back when it wants this tool. Your `call_function` matches on this string.
- **`description`**: the only prose about the tool. The model uses it to decide *whether* this tool fits the question; with ten tools, this is how it picks one. Here the docstring and the description say different things, and only the description affects the model.
- **`parameters`**: a JSON Schema that tells the model which argument names and types to write. `"strict": true` makes the API constrain generation so the arguments always match this schema. It's the same constrained decoding you saw with `response_format` in Exhaustive 2.

##### How this relates to `Field(description=...)`
`Field` descriptions reach the model because the SDK converts your Pydantic class into a JSON Schema and puts it into the request body (you'll see it in Section 6). A plain function gets no such conversion; nothing reads its signature or docstring.

The tool equivalent of `Field(description=...)` is a `"description"` key inside each property. The course code has none, but you could write:
```python
"properties": {
    "latitude": {"type": "number", "description": "Latitude in decimal degrees, e.g. 59.91 for Oslo."},
    "longitude": {"type": "number", "description": "Longitude in decimal degrees, e.g. 10.75 for Oslo."},
},
```

Two related facts, so the rule doesn't get over-simplified:
- A docstring on a **Pydantic class** *is* sent, because Pydantic copies it into the schema as the top-level `"description"`. If you add `"""Final answer about the weather."""` under `class WeatherResponse(BaseModel):`, the schema the SDK sends starts with `{'description': 'Final answer about the weather.', 'properties': {...`.
- Some agent frameworks (e.g. the OpenAI Agents SDK's `@function_tool`) build the tool definition *from* a function's signature and docstring. The docstring reaches the model there only because the framework copies it into `"description"`. The plain `OpenAI()` client used here doesn't do that.

**The rule behind both:** the model only sees what is in the request body, so something has to copy text into it. Pydantic does that for a class (it writes the class into the schema), and `@function_tool` does it for a function. With a plain function and the plain client, nothing does: `get_weather` itself is never passed to the API, only the `tools` dict you wrote by hand.

In [7]:
messages = [
    {"role": "system", "content": "You are a helpful weather assistant."},
    {"role": "user", "content": "What's the weather like in Oslo today?"},
]

completion = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
)

In [10]:
print(type(completion))
print(completion.model_dump_json(indent=2))

<class 'openai.types.chat.chat_completion.ChatCompletion'>
{
  "id": "chatcmpl-EPjMoBPyvkoZAPAP9czmew4GPDHLB",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": [
          {
            "id": "call_jVJ67G2TzgPRcCCApHW4ya7r",
            "function": {
              "arguments": "{\"latitude\":59.9139,\"longitude\":10.7522}",
              "name": "get_weather"
            },
            "type": "function"
          }
        ]
      }
    }
  ],
  "created": 1789801298,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 289,
    "prompt_tokens": 150,
    "total_tokens": 439,
    "complet

In [ ]:
pprint(completion.model_dump(), sort_dicts=False)   #"pretty print"

{'id': 'chatcmpl-EPjMoBPyvkoZAPAP9czmew4GPDHLB',
 'choices': [{'finish_reason': 'tool_calls',
              'index': 0,
              'logprobs': None,
              'message': {'content': None,
                          'refusal': None,
                          'role': 'assistant',
                          'annotations': [],
                          'audio': None,
                          'function_call': None,
                          'tool_calls': [{'id': 'call_jVJ67G2TzgPRcCCApHW4ya7r',
                                          'function': {'arguments': '{"latitude":59.9139,"longitude":10.7522}',
                                                       'name': 'get_weather'},
                                          'type': 'function'}]}}],
 'created': 1789801298,
 'model': 'gpt-5-nano-2025-08-07',
 'object': 'chat.completion',
 'metadata': None,
 'moderation': None,
 'service_tier': 'default',
 'system_fingerprint': None,
 'usage': {'completion_tokens': 289,
           'prompt

#### 2. Reading the response: the model *asked* for a function, and nothing ran

**Key point:** the model's entire reply to call 1 is the request "run `get_weather` with `{"latitude":59.9139,"longitude":10.7522}`". There is no answer text (`content=None`), and `finish_reason='tool_calls'` says the model stopped because it wants a tool run. Everything else in the output is bookkeeping.

The two outputs above are **the same data in two forms**. Both come from Pydantic methods, which exist here because `ChatCompletion` is a Pydantic `BaseModel`:
- `completion.model_dump_json(indent=2)` returns a **string** of JSON (`null`, double quotes). `indent=2` puts one field per line.
- `completion.model_dump()` returns a Python **dict** (`None`, single quotes). `pprint` ("pretty print", from Python's standard library, imported at the top) prints it one key per line, and `sort_dicts=False` keeps the original key order instead of sorting the keys alphabetically.

A plain `print(completion)` would print the object in its own form, `ChatCompletion(id='chatcmpl-...', choices=[Choice(...)], ...)`, all on one line. The class names from that form are the ones in brackets in the tree below.

##### The tree (only the parts that matter; the rest are `None` or empty here)

<pre style="font-size: 0.85em; line-height: 1.4; white-space: pre; overflow-x: auto;">
completion  <span style="opacity: 0.55">(ChatCompletion)</span>
├─ id        'chatcmpl-EPjMoBPyvkoZAPAP9czmew4GPDHLB'
├─ created   1789801298                 <span style="opacity: 0.7">← Unix time</span>
├─ model     'gpt-5-nano-2025-08-07'    <span style="opacity: 0.7">← exact model version</span>
├─ choices   <span style="opacity: 0.55">(list, 1 item)</span>
│  └─ [0]  <span style="opacity: 0.55">(Choice)</span>
│     ├─ finish_reason  'tool_calls'    <span style="opacity: 0.7">← it wants a tool run</span>
│     └─ message  <span style="opacity: 0.55">(ChatCompletionMessage)</span>
│        ├─ role        'assistant'
│        ├─ content     None            <span style="opacity: 0.7">← no answer text</span>
│        └─ tool_calls  <span style="opacity: 0.55">(list, 1 item)</span>
│           └─ [0]  <span style="opacity: 0.55">(ChatCompletionMessageFunctionToolCall)</span>
│              ├─ id        'call_jVJ67G2TzgPRcCCApHW4ya7r'
│              ├─ type      'function'
│              └─ function  <span style="opacity: 0.55">(Function)</span>
│                 ├─ name       'get_weather'
│                 └─ arguments  '{"latitude":59.9139,"longitude":10.7522}'
└─ usage     <span style="opacity: 0.55">(CompletionUsage)</span>
   ├─ prompt_tokens      150            <span style="opacity: 0.7">← tokens read</span>
   ├─ completion_tokens  289            <span style="opacity: 0.7">← tokens written (256 reasoning)</span>
   └─ total_tokens       439
</pre>

The same path reaches the arguments string `'{"latitude":59.9139,"longitude":10.7522}'` in both forms:
```python
# object: dots
completion.choices[0].message.tool_calls[0].function.arguments

# dict: keys
d = completion.model_dump()
d["choices"][0]["message"]["tool_calls"][0]["function"]["arguments"]
```

Things worth noticing:
- **`finish_reason`** is `'tool_calls'` here. For a normal text answer it's `'stop'`.
- **`arguments` is in quotes:** it's a **string** that contains JSON, not a dict. Section 4 turns it into a dict with `json.loads`.
- **The tool call's `id`** (`'call_jVJ67G2TzgPRcCCApHW4ya7r'`) is how the result you send back gets paired with this request (Section 4).
- **Where the coordinates came from:** nothing in the request contained Oslo's latitude and longitude. The model supplied them from its training knowledge. That is the part of the job the model does: choosing the tool and filling in its arguments.
- **Reasoning tokens:** `gpt-5-nano` is a reasoning model, so it thinks privately before writing. Of the 289 tokens it wrote, 256 were that hidden reasoning (`usage.completion_tokens_details.reasoning_tokens`), and only the remaining 33 were the tool call itself. You pay for all 289.
- **The `None`/empty fields** belong to features not used here: `logprobs`, `audio`, `refusal` (filled if the model declines), `annotations` (e.g. web-search citations), and `function_call` (the older, deprecated form of `tool_calls`).

##### Walking those paths in code
The cell below reads exactly those fields off the object, with the dotted form. `tool_calls` is a list, so the loop is what handles a reply that asks for two tools at once. `repr()` prints the quotes, which is what makes `arguments` visibly a string rather than a dict, and `type(...).__name__` says so outright.

In [ ]:
message = completion.choices[0].message

print("finish_reason:", completion.choices[0].finish_reason)
print("content:      ", message.content)
for tool_call in message.tool_calls:
    print("tool call:")
    print("   id:        ", tool_call.id)
    print("   name:      ", tool_call.function.name)
    print("   arguments: ", repr(tool_call.function.arguments), f"({type(tool_call.function.arguments).__name__})")

finish_reason: tool_calls
content:       None
tool call:
   id:         call_jVJ67G2TzgPRcCCApHW4ya7r
   name:       get_weather
   arguments:  '{"latitude":59.9139,"longitude":10.7522}' (str)


In [12]:
messages

[{'role': 'system', 'content': 'You are a helpful weather assistant.'},
 {'role': 'user', 'content': "What's the weather like in Oslo today?"}]

#### 3. Call 1 did not fetch the weather, so why the rest?

**Key point:** `tools` + `messages` + `create()` only get the model to *choose* a tool and *write its arguments*. They can't fetch anything, because the model runs on OpenAI's servers with no access to your function and can't execute Python. The weather is fetched by your code in the next cell. Then a second API call is needed so the model can read the result and write the answer.

Three pieces of evidence from the outputs above:
1. `content=None`: the model wrote no answer, only a tool request.
2. `messages` (the cell just above) still holds only the 2 messages you wrote. `create()` doesn't add anything to your list, and the API keeps no memory between calls. Whatever the model should see next time, *you* have to append.
3. Open-Meteo hasn't been contacted yet. The only code that sends a request to `api.open-meteo.com` is the body of `get_weather`. It runs on your machine, and nothing has called it so far.

So the roles are:
- **Completion 1** decides *which* function to run and *with which arguments*. It does not retrieve the information.
- **Your code** (the next cell) retrieves the information.
- **Completion 2** reads that information and writes the answer, here in the `WeatherResponse` shape.

#### 4. Running the tool ourselves: `call_function`, `**args`, and the loop

##### `call_function` is a dispatcher
**What it is:** a function that takes the tool name the model wrote and runs the matching Python function.

**Why it's needed:** the model only gives you the *string* `'get_weather'`, and you can't call a string. `call_function` maps the name to the real function. With one tool it's a single `if`. With several tools you add one branch per tool (or use a dict like `{"get_weather": get_weather}`).

##### `**args` unpacks a dict into keyword arguments
**What it does:** `**` inside a function call takes each `key: value` pair of a dict and passes it as `key=value`.

This checks that `json.loads` turns the model's arguments string into a dict:
```python
arguments = '{"latitude":59.9139,"longitude":10.7522}'
print(type(arguments))
args = json.loads(arguments)
print(args)
print(type(args))
```
```
<class 'str'>
{'latitude': 59.9139, 'longitude': 10.7522}
<class 'dict'>
```

This checks that `get_weather(**args)` is the same call as writing the keywords out by hand. It uses a stub `get_weather` that returns its inputs instead of calling the API:
```python
def get_weather(latitude, longitude):
    return f"called with latitude={latitude}, longitude={longitude}"

print(get_weather(**args))
print(get_weather(latitude=59.9139, longitude=10.7522))
```
```
called with latitude=59.9139, longitude=10.7522
called with latitude=59.9139, longitude=10.7522
```

This works only because the property names in the tool's `parameters` (`latitude`, `longitude`) exactly match the Python parameter names. This checks what happens when they don't:
```python
get_weather(**{"lat": 59.9139, "lon": 10.7522})
```
```
TypeError: get_weather() got an unexpected keyword argument 'lat'
```

##### The loop, line by line
`completion.choices[0].message.tool_calls` is a **list**, because the model can ask for several calls in one reply (e.g. "weather in Oslo and Paris?" gives two `get_weather` calls). The loop handles each one.

Before the loop, once:
- `messages.append(completion.choices[0].message)` adds the model's tool request (the whole assistant message from call 1) to the history. The API requires every `"role": "tool"` message to come after an assistant message whose `tool_calls` contains the matching id. Without it, call 2 would contain a result answering no request, and the API rejects that.

Inside the loop, once per tool call:
- `name = tool_call.function.name` gives `'get_weather'`, which your code uses to pick the function.
- `args = json.loads(tool_call.function.arguments)` turns the string into a dict (shown above) so Python can use it.
- `result = call_function(name, args)` is **where `get_weather` actually runs** and the real HTTP request goes to Open-Meteo. In the saved run (before the Section 7 fix) it returned `{'time': '2026-09-19T07:15', 'interval': 900, 'temperature_2m': 14.5, 'wind_speed_10m': 14.4}`. Now that dict comes back under `"current"`, next to `"units"` and `"timezone"`.
- `messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)})` hands the result back:
  - `"role": "tool"` marks the message as a tool result, not something the user said.
  - `"tool_call_id"` pairs the result with the request's id (`call_jVJ6...`). With several calls, this tells the model which result answers which request.
  - `json.dumps(result)` turns the dict into a string, because a message's `content` must be text.

The two conversions go in opposite directions: `json.loads` converts **string → dict** for your code, and `json.dumps` converts **dict → string** for the model.

##### Why the first append is above the loop (changed from the course code)
**Key point:** the model's request is **one message that can hold several tool calls**, not one message per call. The course code appends that single message once per pass of the loop, so it lands in the history twice, and call 2 is rejected.

I ran both versions against the real API with the question "What's the weather like in Oslo and in Paris today?". The model replied with **one** assistant message carrying **two** tool calls:

```
assistant message (one reply)
├─ tool_calls[0]   id-Oslo    get_weather(59.91, 10.75)
└─ tool_calls[1]   id-Paris   get_weather(48.85, 2.35)
```
The real ids are long strings; `id-Oslo` and `id-Paris` stand in for them here. The loop then runs twice, once per tool call. In **both** versions `get_weather` runs exactly twice and both cities are fetched correctly — the difference is only in what gets appended to `messages`.

**Version A — the course code: the append sits INSIDE the loop**
```python
for tool_call in completion.choices[0].message.tool_calls:
    messages.append(completion.choices[0].message)   # ← runs on every pass
    ...
    messages.append({"role": "tool", ...})
```
The history it builds, 6 messages:
```
1  system
2  user
3  assistant   asks for BOTH:  id-Oslo + id-Paris     ← appended on pass 1
4  tool        answers id-Oslo
5  assistant   asks for BOTH:  id-Oslo + id-Paris     ← the SAME message again, pass 2
6  tool        answers id-Paris
```
Messages 3 and 5 are the same object appended twice. Oslo is not looked up twice; the duplicate is the model's *request*, not the weather call.

Why the API refuses it: an assistant message carrying `tool_calls` must be followed by a tool message for **every** id it contains, before anything else appears. Message 3 asks for two results, but only message 4 (Oslo) follows before message 5 interrupts, so `id-Paris` is left unanswered. Call 2 came back as a 400, naming the Paris id:
> An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_UEUm2GlZIM5muyhnnBa0Fjbs

**Version B — this notebook: the append sits BEFORE the loop**
```python
messages.append(completion.choices[0].message)       # ← runs once
for tool_call in completion.choices[0].message.tool_calls:
    ...
    messages.append({"role": "tool", ...})
```
The history it builds, 5 messages:
```
1  system
2  user
3  assistant   asks for BOTH:  id-Oslo + id-Paris
4  tool        answers id-Oslo
5  tool        answers id-Paris
```
One request, then one answer per id directly behind it, each tagged with the id it belongs to. Call 2 was accepted and the model wrote its answer (`finish_reason` `'stop'`).

With a single tool call the two versions produce **identical** histories, which is why the course code works for the Oslo-only question in this notebook. The bug only surfaces when the model asks for two or more tools in one reply, and parallel tool calls are on by default.

In [13]:
def call_function(name, args):
    if name == "get_weather":
        return get_weather(**args)


messages.append(completion.choices[0].message)  # the model's request: once, not once per tool call

for tool_call in completion.choices[0].message.tool_calls:
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)

    result = call_function(name, args)
    messages.append(
        {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}
    )

In [14]:
messages

[{'role': 'system', 'content': 'You are a helpful weather assistant.'},
 {'role': 'user', 'content': "What's the weather like in Oslo today?"},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_jVJ67G2TzgPRcCCApHW4ya7r', function=Function(arguments='{"latitude":59.9139,"longitude":10.7522}', name='get_weather'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_jVJ67G2TzgPRcCCApHW4ya7r',
  'content': '{"time": "2026-09-19T07:15", "interval": 900, "temperature_2m": 14.5, "wind_speed_10m": 14.4}'}]

#### 5. The four messages the second call receives

**Key point:** `messages` is the model's entire memory. Call 2 reads all four entries, and entry 4 is the only place the weather data exists.

1. `system`: written by you.
2. `user`: written by you.
3. `assistant`: written by the model in call 1 and appended by you. It's a `ChatCompletionMessage` object, not a dict; the SDK accepts that and converts it to JSON when sending.
4. `tool`: written by you. It holds `get_weather`'s result as a string, and its `tool_call_id` matches the id in entry 3.

This is the `messages` part of call 2's request body, captured the same way as in Section 1:

<div style="font-size: 0.85em">

```json
[
  {"role": "system", "content": "You are a helpful weather assistant."},
  {"role": "user", "content": "What's the weather like in Oslo today?"},
  {
    "content": null,
    "role": "assistant",
    "tool_calls": [
      {
        "id": "call_jVJ67G2TzgPRcCCApHW4ya7r",
        "function": {
          "arguments": "{\"latitude\":59.9139,\"longitude\":10.7522}",
          "name": "get_weather"
        },
        "type": "function"
      }
    ]
  },
  {
    "role": "tool",
    "tool_call_id": "call_jVJ67G2TzgPRcCCApHW4ya7r",
    "content": "{\"time\": \"2026-09-19T07:15\", \"interval\": 900, \"temperature_2m\": 14.5, \"wind_speed_10m\": 14.4}"
  }
]
```

</div>

The `\"` are escaped quotes: the arguments and the tool result are strings that contain JSON, sitting inside the outer JSON.

The SDK sends only the fields that were set on the entry-3 object. Depending on what the API returned, empty fields like `"refusal": null` can also appear; they carry no information.

In [15]:
class WeatherResponse(BaseModel):
    temperature: float = Field(
        description="The current temperature in celsius for the given location."
    )
    response: str = Field(
        description="A natural language response to the user's question."
    )


#### 6. Call 2: the model writes the answer, and `Field()` descriptions get read

**Key point:** call 2 sends the four messages, the same `tools`, and the `WeatherResponse` schema. The model now has the weather data (entry 4), so it writes the final answer instead of asking for a tool.

This is the `response_format` part of call 2's request body. It's how your `Field(description=...)` text reaches the model:

<div style="font-size: 0.85em">

```json
"response_format": {
  "type": "json_schema",
  "json_schema": {
    "schema": {
      "properties": {
        "temperature": {
          "description": "The current temperature in celsius for the given location.",
          "title": "Temperature",
          "type": "number"
        },
        "response": {
          "description": "A natural language response to the user's question.",
          "title": "Response",
          "type": "string"
        }
      },
      "required": ["temperature", "response"],
      "title": "WeatherResponse",
      "type": "object",
      "additionalProperties": false
    },
    "name": "WeatherResponse",
    "strict": true
  }
}
```

</div>

**Why `tools=tools` is passed again:** the API stores nothing between calls, so each request has to describe everything again. If you want the model to still know `get_weather` exists (and be able to call it again), its definition has to be in this request too.

**What the code assumes:** call 2 could come back with *another* tool request instead of an answer. Then `finish_reason` would be `'tool_calls'` and `.parsed` would be `None`. This course code assumes one round trip is enough. Real agents repeat steps 2 and 3 of the flow in a loop until `finish_reason` is `'stop'`.

In [16]:
completion_2 = client.chat.completions.parse(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
    response_format=WeatherResponse,
)

In [17]:
final_response = completion_2.choices[0].message.parsed
print(final_response.temperature)
print(final_response.response)

14.5
The current temperature in Oslo is 14.5°C as of 07:15 local time, with winds around 14 m/s.


#### 7. Check the answer against what the model was given

**Key point:** the model only knows what is in `messages`. In the saved run the tool message carried bare numbers with no units and no timezone, so the model guessed both, and got both wrong.

##### First: where the four fields in the tool message come from
Section 5 shows the tool message holding this string:
```
{"time": "2026-09-19T07:15", "interval": 900, "temperature_2m": 14.5, "wind_speed_10m": 14.4}
```
Nothing added those four fields to the result. They **are** one block of Open-Meteo's reply, `data["current"]`, and the run that produced this notebook used the course's return line, `return data["current"]`.

Open-Meteo's full reply is much larger. A live request for Oslo (2026-09-20), with the long `hourly` arrays cut short:
```json
{
  "latitude": 59.9, "longitude": 10.75, "elevation": 37.0,
  "utc_offset_seconds": 0,
  "timezone": "GMT",
  "timezone_abbreviation": "GMT",
  "current_units": {"time": "iso8601", "interval": "seconds",
                    "temperature_2m": "°C", "wind_speed_10m": "km/h"},
  "current":       {"time": "2026-09-20T07:45", "interval": 900,
                    "temperature_2m": 13.3, "wind_speed_10m": 19.1},
  "hourly_units":  {"time": "iso8601", "temperature_2m": "°C", ...},
  "hourly":        {"time": ["2026-09-20T00:00", ...], "temperature_2m": [...], ...}
}
```
`data["current"]` is the block labelled `"current"`: those four keys, nothing else. `json.dumps` turned that one dict into the tool message string, which is why the fields sit flat at the top level. `interval: 900` is Open-Meteo's own bookkeeping (the reading refreshes every 900 seconds); it rides along inside the block.

What the course's return line dropped: `current_units`, `timezone`, `hourly_units` and `hourly`.

##### Why the `get_weather` cell above doesn't match the output below it
The `get_weather` cell near the top of this notebook is the **fixed** version. The outputs saved in this notebook come from a run made **before** that fix, when the function was still the course's one-liner. The two shapes:

```python
# course code — produced every saved output in this notebook
return data["current"]
# tool message content:
# '{"time": "2026-09-19T07:15", "interval": 900, "temperature_2m": 14.5, "wind_speed_10m": 14.4}'

# fixed code — the cell at the top of this notebook
return {"current": data["current"], "units": data["current_units"], "timezone": data["timezone"]}
# tool message content (live values for Oslo, 2026-09-20):
# '{"current": {"time": "2026-09-20T09:45", "interval": 900, "temperature_2m": 13.3,
#   "wind_speed_10m": 5.3}, "units": {"time": "iso8601", "interval": "seconds",
#   "temperature_2m": "°C", "wind_speed_10m": "m/s"}, "timezone": "Europe/Oslo"}'
```

The numbers stay where they were, one level deeper, under `"current"`. Re-running this notebook replaces the flat tool message with the nested one; `Supplement_3-tools.ipynb` already holds such a run.

##### What the model got wrong, and why
- **Wind: right number, wrong label.** `"wind_speed_10m": 14.4` arrived with no unit. The course URL requests no unit, and Open-Meteo's default is km/h — the reply says so in `current_units`, which the return line dropped. The model wrote "around 14 m/s". Nothing was miscalculated; the number was simply called by the wrong name, and 14.4 km/h is 4 m/s, so the answer overstates the wind 3.6-fold.
- **Time: right instant, wrong clock.** `"time": "2026-09-19T07:15"` arrived with no timezone. The reply's `timezone` was `"GMT"`, also dropped. Oslo in September runs on UTC+2, so that instant is 09:15 there. The model wrote "07:15 local time".
- **Temperature: right, for an instructive reason.** `temperature_2m` is in °C, and the word "celsius" appears twice in the request: in the tool's `description` and in the `Field` description. That unit reached the model through the **tool definition**, not through the data. The two values with no label anywhere in the request are exactly the two it got wrong.

##### The fix: two halves, both needed
The same URL twice at the same moment, one plain and one with the two extra parameters (live for Oslo, 2026-09-20):

| | course URL | `&timezone=auto&wind_speed_unit=ms` |
| --- | --- | --- |
| `current.time` | `"2026-09-20T07:45"` | `"2026-09-20T09:45"` |
| `current.wind_speed_10m` | `19.1` | `5.3` |
| `current_units.wind_speed_10m` | `"km/h"` | `"m/s"` |
| `timezone` | `"GMT"` | `"Europe/Oslo"` |
| `current.temperature_2m` | `13.3` | `13.3` |

Both columns describe the same weather: 19.1 km/h **is** 5.3 m/s, and 07:45 GMT **is** 09:45 in Oslo.

1. **The URL parameters change which numbers arrive.** On their own they fix nothing for the model, because `current` still carries no labels. The model would still be guessing; it would just happen to guess right.
2. **Returning `current_units` and `timezone` puts the labels into the tool message.** That is what removes the guess.

One half makes the values what you want, the other tells the model what they are. Same lesson as Section 1: if the model should know something, it has to be in the request.